In [7]:
# BRONZE: Đọc dữ liệu thô từ Volume
file_path = "/Volumes/workspace/default/ecommerce_raw/Brazilian E-Commerce Public Dataset by Olist.csv"

# Đọc CSV, giữ dữ liệu gốc
df_raw = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

# Xem nhanh 10 dòng
display(df_raw.limit(10))

NameError: name 'spark' is not defined

In [6]:
# SILVER: Làm sạch dữ liệu cơ bản
# Bỏ dòng trùng và dòng rỗng hoàn toàn
df_clean = df_raw.dropDuplicates().dropna(how="all")

# Đường dẫn lưu Delta Silver
delta_table_path = "/Volumes/workspace/default/ecommerce_raw/ecommerce_silver"

# Ghi ra Delta (ghi đè mỗi lần chạy)
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_table_path)

# Xem lịch sử Delta để kiểm tra
display(spark.sql(f"DESCRIBE HISTORY delta.`{delta_table_path}`"))

NameError: name 'df_raw' is not defined

In [0]:
# GOLD: Tạo bảng dữ liệu sẵn sàng cho dịch vụ
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

silver_path = "/Volumes/workspace/default/ecommerce_raw/ecommerce_silver"
base_gold_path = "/Volumes/workspace/default/ecommerce_raw"

# Đọc dữ liệu Silver
silver_df = spark.read.format("delta").load(silver_path)

# Chuẩn hóa kiểu dữ liệu
work_df = (
    silver_df
    .withColumn("payment_value", F.coalesce(F.col("payment_value").cast("double"), F.lit(0.0)))
    .withColumn("order_purchase_ts", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("order_estimated_ts", F.to_timestamp("order_estimated_delivery_date"))
    .withColumn("order_delivered_ts", F.to_timestamp("order_delivered_customer_date"))
)

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.default")


def save_gold(df, path_name, table_name):
    """Lưu Delta vào Volumes và đăng ký bảng để truy vấn."""
    full_path = f"{base_gold_path}/{path_name}"
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(full_path)
    )
    (
        spark.read.format("delta").load(full_path)
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.default.{table_name}")
    )
    return full_path


# GOLD 1: Sản phẩm nổi bật theo danh mục
top_products_df = (
    work_df
    .filter(F.col("product_id").isNotNull() & F.col("product_category_name").isNotNull())
    .groupBy("product_category_name", "product_id")
    .agg(F.countDistinct("order_id").alias("total_orders"))
)

w_cat = Window.partitionBy("product_category_name").orderBy(F.desc("total_orders"), F.asc("product_id"))
top_products_df = top_products_df.withColumn("rank_in_category", F.dense_rank().over(w_cat))

path_top_products = save_gold(top_products_df, "gold_top_products", "gold_top_products")


# GOLD 2: Cặp sản phẩm mua cùng nhau
order_products = (
    work_df
    .select("order_id", "product_id")
    .filter(F.col("order_id").isNotNull() & F.col("product_id").isNotNull())
    .dropDuplicates()
)

cross_sell_df = (
    order_products.alias("a")
    .join(
        order_products.alias("b"),
        (F.col("a.order_id") == F.col("b.order_id"))
        & (F.col("a.product_id") < F.col("b.product_id")),
        "inner",
    )
    .groupBy(
        F.col("a.product_id").alias("product_id_A"),
        F.col("b.product_id").alias("product_id_B"),
    )
    .agg(F.countDistinct(F.col("a.order_id")).alias("frequency_bought_together"))
    .filter(F.col("frequency_bought_together") >= 2)
    .orderBy(F.desc("frequency_bought_together"))
)

path_cross_sell = save_gold(cross_sell_df, "gold_cross_sell", "gold_cross_sell")


# GOLD 3: Phân khúc khách hàng theo RFM
rfm_base_df = (
    work_df
    .select("customer_unique_id", "order_id", "order_purchase_ts", "payment_value")
    .filter(F.col("customer_unique_id").isNotNull() & F.col("order_id").isNotNull() & F.col("order_purchase_ts").isNotNull())
)

snapshot_date = rfm_base_df.agg(F.max(F.to_date("order_purchase_ts")).alias("max_date")).first()["max_date"]

rfm_df = (
    rfm_base_df
    .groupBy("customer_unique_id")
    .agg(
        F.datediff(F.lit(snapshot_date), F.max(F.to_date("order_purchase_ts"))).alias("recency"),
        F.countDistinct("order_id").alias("frequency"),
        F.round(F.sum("payment_value"), 2).alias("monetary"),
    )
)

rfm_df = (
    rfm_df
    .withColumn(
        "r_score",
        F.when(F.col("recency") <= 30, 5)
        .when(F.col("recency") <= 60, 4)
        .when(F.col("recency") <= 120, 3)
        .when(F.col("recency") <= 180, 2)
        .otherwise(1),
    )
    .withColumn(
        "f_score",
        F.when(F.col("frequency") >= 10, 5)
        .when(F.col("frequency") >= 6, 4)
        .when(F.col("frequency") >= 3, 3)
        .when(F.col("frequency") >= 2, 2)
        .otherwise(1),
    )
    .withColumn(
        "m_score",
        F.when(F.col("monetary") >= 3000, 5)
        .when(F.col("monetary") >= 1500, 4)
        .when(F.col("monetary") >= 700, 3)
        .when(F.col("monetary") >= 200, 2)
        .otherwise(1),
    )
    .withColumn("rfm_score", F.col("r_score") + F.col("f_score") + F.col("m_score"))
    .withColumn(
        "customer_segment",
        F.when((F.col("rfm_score") >= 13) & (F.col("recency") <= 60), F.lit("VIP"))
        .when(F.col("rfm_score") >= 10, F.lit("Loyal"))
        .when(F.col("rfm_score") >= 7, F.lit("Potential"))
        .when(F.col("recency") > 180, F.lit("At Risk"))
        .otherwise(F.lit("Regular")),
    )
    .select("customer_unique_id", "recency", "frequency", "monetary", "rfm_score", "customer_segment")
)

path_user_segments = save_gold(rfm_df, "gold_user_segments", "gold_user_segments")


# GOLD 4: Doanh thu theo khu vực
geo_revenue_df = (
    work_df
    .select("customer_state", "customer_city", "order_id", "payment_value")
    .filter(F.col("customer_state").isNotNull() & F.col("customer_city").isNotNull() & F.col("order_id").isNotNull())
    .groupBy("customer_state", "customer_city")
    .agg(
        F.round(F.sum("payment_value"), 2).alias("total_revenue"),
        F.round(F.sum("payment_value") / F.countDistinct("order_id"), 2).alias("avg_order_value"),
        F.countDistinct("order_id").alias("total_orders"),
    )
    .orderBy(F.desc("total_revenue"))
)

path_geo_revenue = save_gold(geo_revenue_df, "gold_geo_revenue", "gold_geo_revenue")


# GOLD 5: Hiệu quả giao hàng
shipping_df = (
    work_df
    .select("order_id", "order_estimated_ts", "order_delivered_ts")
    .filter(F.col("order_id").isNotNull())
    .dropDuplicates(["order_id"])
    .withColumn("delay_days", F.datediff(F.col("order_delivered_ts"), F.col("order_estimated_ts")))
    .withColumn(
        "shipping_status",
        F.when(F.col("order_delivered_ts").isNull(), F.lit("not_delivered"))
        .when(F.col("delay_days") <= 0, F.lit("on_time"))
        .otherwise(F.lit("delayed")),
    )
    .select("order_id", "shipping_status", "delay_days")
)

path_shipping = save_gold(shipping_df, "gold_shipping_performance", "gold_shipping_performance")


# GOLD 6: Thống kê phương thức thanh toán
payment_stats_df = (
    work_df
    .select("payment_type", "order_id", "payment_value")
    .filter(F.col("payment_type").isNotNull() & F.col("order_id").isNotNull())
    .groupBy("payment_type")
    .agg(
        F.round(F.sum("payment_value"), 2).alias("total_value"),
        F.countDistinct("order_id").alias("count_orders"),
        F.round(F.avg("payment_value"), 2).alias("avg_payment_value"),
    )
    .orderBy(F.desc("total_value"))
)

path_payment = save_gold(payment_stats_df, "gold_payment_stats", "gold_payment_stats")


# Minh chứng cho báo cáo
display(top_products_df.orderBy("product_category_name", "rank_in_category").limit(20))
display(cross_sell_df.limit(20))
display(rfm_df.limit(20))
display(geo_revenue_df.limit(20))
display(shipping_df.limit(20))
display(payment_stats_df.limit(20))

display(spark.sql("SHOW TABLES IN workspace.default"))
display(DeltaTable.forPath(spark, path_top_products).history())

product_category_name,product_id,total_orders,rank_in_category
agro_industria_e_comercio,11250b0d4b709fee92441c5f34122aed,21,1
agro_industria_e_comercio,423a6644f0aa529e8828ff1f91003690,18,2
agro_industria_e_comercio,672e757f331900b9deea127a2a7b79fd,14,3
agro_industria_e_comercio,3bebad3cf2c8d1a8d3ce97174643e054,12,4
agro_industria_e_comercio,a0fe1efb855f3e786f0650268cd77f44,12,5
agro_industria_e_comercio,16d096faa27582985f849f08370cf1ed,8,6
agro_industria_e_comercio,b5aebb467d9a92162173cbd234e00d99,8,7
agro_industria_e_comercio,1f541a93f4fa4b8d15c7d5ed83836484,3,8
agro_industria_e_comercio,6ff1fc9209c7854704a4f75c9fac41b4,3,9
agro_industria_e_comercio,980ecbcc15fe174ec1e5757c4d75b1bf,3,10


product_id_A,product_id_B,frequency_bought_together
36f60d45225e60c7da4558b070ce4b60,e53e557d5a159f5aa2c5e995dfdf244b,34
35afc973633aaeb6b877ff57b2793310,99a4788cb24856965c36a24e339b6058,29
4fcb3d9a5f4871e8362dfedbdb02b064,f4f67ccaece962d013a4e1d7dc3a61f7,17
36f60d45225e60c7da4558b070ce4b60,3f14d740544f37ece8a9e7bc8349797e,12
389d119b48cf3043d311335e499d9c6b,422879e10f46682990de24d770e7f83d,11
389d119b48cf3043d311335e499d9c6b,53759a2ecddad2bb87a079a1f1519f73,9
368c6c730842d78016ad823897a372db,53759a2ecddad2bb87a079a1f1519f73,8
422879e10f46682990de24d770e7f83d,53759a2ecddad2bb87a079a1f1519f73,7
5d790355cbeded0cd60e25cbc4c527a2,5fc3e6a4b52b0c414458104ed4037f1c,6
060cb19345d90064d1015407193c233d,98d61056e0568ba048e5d78038790e77,6


customer_unique_id,recency,frequency,monetary,rfm_score,customer_segment
d2d29ce9d7fc1b0e7415d1f35fd019b3,256,1,368.11,4,At Risk
982cd72d36a006db9ea1001555fc5f8a,14,1,176.88,7,Potential
bf0a805b588b889511e15496905cf7c5,457,1,641.58,4,At Risk
6e7bfbc2f1b96b9be02a19a233d98e0f,1,1,66.67,7,Potential
9749e3b7d02a7194473d5a6784071186,359,1,310.28,4,At Risk
ee5972781a5a53c05b50850f85d63ffc,64,1,171.2,5,Regular
876b7343b50c63c8f32b59908a9e554d,390,1,188.96,3,At Risk
0af92d2ac166e21f4c91e660504a5d47,413,1,65.74,3,At Risk
e957c1e47f8592055a8f19fd2035ec29,203,1,43.0,3,At Risk
0b2d0021b09dd4791c0010d4be6e1760,56,1,103.26,6,Regular


customer_state,customer_city,total_revenue,avg_order_value,total_orders
SP,sao paulo,2714913.35,183.3,14811
RJ,rio de janeiro,1505895.8,231.75,6498
MG,belo horizonte,483701.77,182.53,2650
DF,brasilia,415948.91,203.7,2042
PR,curitiba,319988.23,217.09,1474
RS,porto alegre,290585.28,219.64,1323
BA,salvador,274564.3,234.67,1170
SP,campinas,258256.35,185.66,1391
SP,guarulhos,197586.17,175.17,1128
GO,goiania,191265.71,298.85,640


order_id,shipping_status,delay_days
a0fa357673549db3161966af6bda1598,on_time,-8
a4cfd47a44ceac0a7ebb64dd03f2a4a6,on_time,-17
5137d2621656ff029745600e991f1835,delayed,18
1a611328643ae11146ba09a4425d2e12,on_time,-19
186e57039f508947e4bdc91cf933b1bf,on_time,-18
813e757cecde8eee7af8d87fea8e50f1,on_time,-10
80e57a9aecdddc40ad34d34dda39838e,on_time,-10
2ff5a9a440a7153c1eb3651f16f631c4,on_time,-10
a7fdfd7d7ae7a53a8232d6d0d7db215b,on_time,-12
02cda26d2cd43c51fefac9e3291fbc9b,on_time,-19


payment_type,total_value,count_orders,avg_payment_value
credit_card,1.501195295E7,73267,179.34
boleto,3886498.68,18926,176.28
voucher,389060.48,3625,64.71
debit_card,242991.23,1458,149.53


database,tableName,isTemporary
default,gold_cross_sell,false
default,gold_geo_revenue,false
default,gold_payment_stats,false
default,gold_recommendations,false
default,gold_shipping_performance,false
default,gold_top_products,false
default,gold_user_segments,false


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-04-24T04:27:07.000Z,76768867430497,23703471.khoa@student.iuh.edu.vn,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(751682238159148),7689fb62-e7c9-46ef-8f97-b2bc9d4ecbed,0424-033256-4jn28jk9-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1043451, numDeletionVectorsRemoved -> 0, numOutputRows -> 31625, numOutputBytes -> 1043451)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-04-24T04:21:24.000Z,76768867430497,23703471.khoa@student.iuh.edu.vn,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(751682238159148),ea218f57-a5d8-4e2c-8adb-f62ee7c917a0,0424-033256-4jn28jk9-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 31625, numOutputBytes -> 1043451)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


Gold layer created successfully with 6 output tables
- Volumes path: /Volumes/workspace/default/ecommerce_raw/gold_top_products -> table workspace.default.gold_top_products
- Volumes path: /Volumes/workspace/default/ecommerce_raw/gold_cross_sell -> table workspace.default.gold_cross_sell
- Volumes path: /Volumes/workspace/default/ecommerce_raw/gold_user_segments -> table workspace.default.gold_user_segments
- Volumes path: /Volumes/workspace/default/ecommerce_raw/gold_geo_revenue -> table workspace.default.gold_geo_revenue
- Volumes path: /Volumes/workspace/default/ecommerce_raw/gold_shipping_performance -> table workspace.default.gold_shipping_performance
- Volumes path: /Volumes/workspace/default/ecommerce_raw/gold_payment_stats -> table workspace.default.gold_payment_stats
